In [1]:
from datasets import load_dataset

from data_processing import _select_usable_examples, process_one_obj_one_band_train_heldout
from singleGP_model import fit_basic_gp
from evaluation_metrics import (
    evaluate_heldout_metrics,
    summarize_single_band_gp_class_metrics,
)

dset_plasticc = load_dataset(
    "MultimodalUniverse/plasticc",
    streaming=True,
    split="train",
).with_format("numpy")

/Users/green/Downloads/multi_outputGP/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Single-band GP
This test aggregates single-band GP evaluation metrics by object class using object-weighted aggregation.

In [2]:
selected_examples, scanned_examples = _select_usable_examples(
    iter(dset_plasticc),
    target_band="r",
    n_objects=500,
    max_examples_to_scan=1000,
    min_points=8,
)

object_results = []

for object_idx, example in enumerate(selected_examples):
    train_data, heldout_data = process_one_obj_one_band_train_heldout(
        example,
        target_band="r",
        normalize_flux=True,
        min_points=8,
        heldout_fraction=0.2,
        min_train_points=5,
        min_heldout_points=1,
        random_state=object_idx,
        strategy="random",
        force_peak_in_train=True,
        peak_mode="absolute",
        peak_alignment="target_abs_peak",
        subtract_background=False,
        scale_mode="local_peak",
    )

    if train_data is None or heldout_data is None:
        continue

    gp = fit_basic_gp(train_data, kernel_type="matern")

    metrics = evaluate_heldout_metrics(
        gp,
        heldout_data,
        train_data=train_data,
        object_data=example,
    )

    metrics["obj_id"] = train_data["obj_id"]
    metrics["obj_type"] = train_data["obj_type"]
    object_results.append(metrics)

class_summary = summarize_single_band_gp_class_metrics(
    object_results,
    output_path="single_band_gp_class_summary.csv",
    print_table=True,
)

class_summary

/Users/green/Downloads/multi_outputGP/.venv/lib/python3.13/site-packages/sklearn/gaussian_process/kernels.py:440: ConvergenceWarning: The optimal value found for dimension 0 of parameter k2__length_scale is close to the specified lower bound 0.05. Decreasing the bound and calling fit again may find a better value.
  warnings.warn(
/Users/green/Downloads/multi_outputGP/.venv/lib/python3.13/site-packages/sklearn/gaussian_process/kernels.py:440: ConvergenceWarning: The optimal value found for dimension 0 of parameter k2__length_scale is close to the specified lower bound 0.05. Decreasing the bound and calling fit again may find a better value.
  warnings.warn(
/Users/green/Downloads/multi_outputGP/.venv/lib/python3.13/site-packages/sklearn/gaussian_process/kernels.py:440: ConvergenceWarning: The optimal value found for dimension 0 of parameter k2__length_scale is close to the specified lower bound 0.05. Decreasing the bound and calling fit again may find a better value.
  warnings.warn(
/

        class  n_objects  n_objects_skipped_small_flux  min_object_flux_scale  \
0         AGN         27                             0               0.000001   
1          EB         24                             0               0.000001   
2          KN          9                             1               0.000001   
3     M-dwarf         39                             0               0.000001   
4         RRL          5                             0               0.000001   
5      SLSN-I         15                             0               0.000001   
6        SNII         95                             0               0.000001   
7        SNIa        182                             0               0.000001   
8   SNIa-91bg         13                             0               0.000001   
9       SNIax         13                             0               0.000001   
10      SNIbc         35                             0               0.000001   
11        TDE         42    

,class,n_objects,n_objects_skipped_small_flux,min_object_flux_scale,n_target_train_mean,n_target_train_std,n_target_train_median,n_target_train_min,n_target_train_max,n_target_train_p10,...,coverage_2sigma_std,coverage_3sigma_mean,coverage_3sigma_std,z_mean_mean,z_mean_std,z_std_mean,z_std_std,n_objects_zstd_n_test_ge_5,z_std_mean_n_test_ge_5,z_std_std_n_test_ge_5
0,AGN,27,0,0.000001,23.925926,10.244876,19.0,13,41,16.0,...,0.142121,0.973513,0.074211,0.066053,0.542859,1.116426,0.416876,24,1.139654,0.435277
1,EB,24,0,0.000001,31.416667,10.816333,41.0,14,41,17.6,...,0.116085,0.925253,0.112977,-0.056863,0.819696,1.448252,1.326105,23,1.484016,1.343248
2,KN,9,1,0.000001,17.777778,1.930905,17.0,15,20,15.0,...,0.062854,1.000000,0.000000,0.066111,0.360350,0.767729,0.376475,7,0.787137,0.398382
3,M-dwarf,39,0,0.000001,31.538462,10.807989,41.0,17,41,18.0,...,0.111383,0.973427,0.084064,0.145183,0.736402,1.158876,2.369522,39,1.158876,2.369522
4,RRL,5,0,0.000001,36.000000,10.000000,41.0,16,41,26.0,...,0.118615,0.836364,0.145455,-0.003392,0.627286,1.961734,0.917412,5,1.961734,0.917412
5,SLSN-I,15,0,0.000001,25.866667,10.800412,20.0,16,41,17.0,...,0.067987,1.000000,0.000000,-0.010036,0.262195,0.639174,0.351310,14,0.673784,0.338033
6,SNII,95,0,0.000001,24.715789,10.086639,20.0,15,41,16.4,...,0.069635,0.993270,0.033213,-0.032579,0.407833,0.724736,0.507659,90,0.741381,0.515559
7,SNIa,182,0,0.000001,27.373626,11.112460,20.0,15,41,16.1,...,0.097981,0.982767,0.062978,-0.125698,0.585290,0.758852,0.731251,174,0.737950,0.678123
8,SNIa-91bg,13,0,0.000001,20.461538,6.295485,19.0,16,41,16.2,...,0.053294,1.000000,0.000000,-0.079941,0.356289,0.514743,0.269819,11,0.584599,0.233042
9,SNIax,13,0,0.000001,30.923077,10.936611,41.0,16,41,19.0,...,0.052331,0.993007,0.024224,0.022483,0.372133,0.691737,0.229266,13,0.691737,0.229266
